# SCF Abstraction: Testing the kernel in closed shells
In this notebook we will test the 4c scf kernel, in order to determine if it works for: 

In [1]:
import numpy as np
from py_mods.src.SCF_4c_dev.types_4c import CS_4c_KU_SCF_Context
from py_mods.src.external.DIRAC_ME import (
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_h5,
    build_uncontracted_basis_from_h5,
)

from py_mods.src.SCF_4c_dev.KUSCF_dev import (
    occupation_4c,
    eri_classified,
    _kuscf_kernel,
)

In [2]:
def scf_state_allocation_from_h5(h5_filename, tot_charge=0):
    S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
    H_nuc = V + W + T
    nuc_charge = get_nuc_charge(h5_filename)
    n_elec = nuc_charge - (tot_charge)

    _, nL, nS = build_uncontracted_basis_from_h5(h5_filename)
    eri = full_eri_from_h5(h5_filename, symmetry=4)
    eri = eri_classified(eri, nL)

    occ_det = occupation_4c(nS, nL, n_elec)

    test_ctx = CS_4c_KU_SCF_Context(
        nL,
        nS,
        S,
        T,
        V,
        W,
        eri,
        n_elec,
        theta=0.00,
        occ=occ_det,
        p_guess="core",
        verbose=True,
        threshold=5e-8,
        acc_hist_size=6,
        acc_iteration_start=5,
        max_iter=500,
    )

    return test_ctx

# 1. $H^-$ Test

the reference energy for $H^-$ is:

$$
E_{H^-} =  -0.4999521765048605
$$



In [3]:
H_cxt = scf_state_allocation_from_h5("data/H_checkpoint.h5", tot_charge=0)
H_kuscf_results = _kuscf_kernel(H_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2           -0.4999521765048612+0.0000000000000000j           -0.4999521765048612+0.0000000000000000j     3.6043E-11
Convergence achieved after 2 iterations.


In [4]:
print(f"H^- energy matches reference: {np.abs(H_kuscf_results.E_SCF - -0.4999521765048605) < 1e-10}")

H^- energy matches reference: True


# 2. $He$ Test

The expected energy after convergence is:

$$
E_{He} = -2.861285099218
$$

In [5]:
He_cxt = scf_state_allocation_from_h5("data/He_checkpoint.h5", tot_charge=0)
He_kuscf_results = _kuscf_kernel(He_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2           -2.7497446361133151+0.0000000000000000j           -2.7497446361133151+0.0000000000000000j     1.3789E+00
    3           -2.8599296846693116-0.0000000000000000j           -0.1101850485559965-0.0000000000000000j     8.5446E-02
    4           -2.8612543623062106+0.0000000000000000j           -0.0013246776368989+0.0000000000000000j     1.0067E-02
    5           -2.8612841684837398+0.0000000000000000j           -0.0000298061775292-0.0000000000000000j     1.7103E-03
    6 

In [6]:
print(f"He energy matches reference: {np.abs(He_kuscf_results.E_SCF - -2.861285099218) < 1e-10}")

He energy matches reference: True


# 3. $Be$ Test

The $Be$ refrerence energy is:

$$
E_{Be} = -14.575569219425365
$$

In [7]:
Be_cxt = scf_state_allocation_from_h5("data/Be_checkpoint.h5", tot_charge=0)
Be_kuscf_results = _kuscf_kernel(Be_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2          -13.7193658574587829+0.0000000000000001j          -13.7193658574587829+0.0000000000000001j     5.1455E+00
    3          -14.5663169029622424-0.0000000000000000j           -0.8469510455034595-0.0000000000000001j     3.6875E-01
    4          -14.5752924211367905+0.0000000000000000j           -0.0089755181745481+0.0000000000000000j     4.9332E-02
    5          -14.5755582600856997+0.0000000000000000j           -0.0002658389489092-0.0000000000000000j     7.5205E-03
    6 

In [8]:
print(f"He energy matches reference: {np.abs(Be_kuscf_results.E_SCF - -14.575569219425365) < 1e-10}")

He energy matches reference: True


# 4. $Ne$ Test

The $Ne$ refrerence energy is:

$$
E_{Ne} = -128.68579576537914
$$

In [9]:
Ne_cxt = scf_state_allocation_from_h5("data/Ne_checkpoint.h5", tot_charge=0)

In [10]:
Ne_kuscf_results = _kuscf_kernel(Ne_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2         -112.4356540807836780-0.0000000000000003j         -112.4356540807836780-0.0000000000000003j     3.6087E+01
    3         -119.6726220088938959+0.0000000000000007j           -7.2369679281102179+0.0000000000000011j     1.0547E+01
    4         -125.9578217480085272-0.0000000000000002j           -6.2851997391146313-0.0000000000000009j     1.1530E+01
    5         -127.5649059806368939-0.0000000000000002j           -1.6070842326283667-0.0000000000000000j     5.7383E+00
    6 

In [11]:
print(f"Ne energy matches reference: {np.abs(Ne_kuscf_results.E_SCF -  -128.68579576537914) < 1e-10}")

Ne energy matches reference: True


# 4. $Mg$ Test 
The reference energy for $Mg$ is:
$$
E_{Mg} =   -199.92997709613846
$$

In [12]:
# Mg_cxt = scf_state_allocation_from_h5("data/Mg_checkpoint.h5", tot_charge=0)
# Mg_kuscf_results = _kuscf_kernel(Mg_cxt)

But this fails probably due to lindep in basis, since running it wont start the scf and raise:

```py
File ~/bin/Oatmeal/py_mods/src/SCF/linalg.py:57, in transformation_matrix(S_munu, method, verbose)
     54     print(transformed)
     56 # Use identity matrix of correct size for check
---> 57 assert np.allclose(
     58     transformed, np.eye(len(S_munu)), atol=1e-7
     59 ), "Transformation failed"
     61 return X

AssertionError: Transformation failed
```

Also even if asking for no lindep removal, DIRAC states in the small component:

```shell
 893    S   A     * Deleted:          3                                       *Smin:   0.73E-09
 894 *** WARNING *** : 3 functions deleted due to numerical linear dependence.
 895 
```

In [13]:
# print(f"Ne energy matches reference: {np.abs(Mg_kuscf_results.E_SCF - -199.92997709613846) < 1e-10}")

# 5. $Ar$ Test 
The reference energy for $Ar$ is:
$$
E_{Ar} =  -528.6631537813
$$

In [14]:
Ar_cxt = scf_state_allocation_from_h5("data/ar_checkpoint.h5", tot_charge=0)

In [15]:
Ar_kuscf_results = _kuscf_kernel(Ar_cxt)

--------------------------------------------------------------------------------------------------------------------------------
|   Iter     |                   E_iter                      |                   Delta_e                   |      norm(e_i)      |
--------------------------------------------------------------------------------------------------------------------------------
    1            0.0000000000000000+0.0000000000000000j            0.0000000000000000+0.0000000000000000j     0.0000E+00
    2         -469.8986476605091980-0.0000000000000017j         -469.8986476605091980-0.0000000000000017j     5.5193E+01
    3         -513.1842083897760176+0.0000000000000012j          -43.2855607292668196+0.0000000000000029j     3.2857E+01
    4         -525.5545929095615065-0.0000000000000018j          -12.3703845197854889-0.0000000000000030j     1.0502E+01
    5         -528.2643980276275215+0.0000000000000000j           -2.7098051180660150+0.0000000000000018j     3.7880E+00
    6 

In [16]:
print(
    f"Ar energy matches reference: {np.abs(Ar_kuscf_results.E_SCF - -528.6631537813) < 1e-10}"
)

Ar energy matches reference: True
